In [ ]:
%pip install pandas scikit-learn matplotlib seaborn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split


In [ ]:
df = pd.read_csv("personal_finance_data.csv")
df.head()


In [ ]:
df = df.dropna()

df.columns = df.columns.str.lower()

print("Dataset Shape:", df.shape)
df.info()


In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["amount"], bins=50)
plt.title("Transaction Amount Distribution")
plt.show()


In [ ]:
category_keywords = {
    "Food": ["restaurant","cafe","pizza","burger","swiggy","zomato"],
    "Transport": ["uber","ola","metro","bus","fuel"],
    "Shopping": ["amazon","flipkart","store","mall"],
    "Bills": ["electricity","water","internet","bill"],
    "Entertainment": ["netflix","spotify","movie","game"],
    "Health": ["pharmacy","hospital","doctor"],
    "Other": []
}

def rule_category(text):
    text = str(text).lower()
    for cat, words in category_keywords.items():
        for w in words:
            if w in text:
                return cat
    return "Other"

df["category"] = df["description"].apply(rule_category)
df.head()


In [ ]:
X = df["description"]
y = df["category"]

vectorizer = TfidfVectorizer()
X_vec = vectorizer.fit_transform(X)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_vec, y)

print("Model trained successfully")


In [ ]:
df["predicted_category"] = model.predict(X_vec)
df.head()


In [ ]:
cat_spend = df.groupby("predicted_category")["amount"].sum()

plt.figure(figsize=(10,5))
cat_spend.plot(kind="bar")
plt.title("Spending by Category")
plt.ylabel("Amount")
plt.show()


In [ ]:
iso = IsolationForest(contamination=0.05, random_state=42)

df["anomaly"] = iso.fit_predict(df[["amount"]])
df["anomaly"] = df["anomaly"].map({1:"Normal",-1:"Unusual"})

df[df["anomaly"]=="Unusual"].head()


In [ ]:
total_spend = df["amount"].sum()

essential = df[df["predicted_category"].isin(
    ["Bills","Health","Transport"]
)]["amount"].sum()

score = max(0, 100 - (essential/total_spend)*100)

print("Financial Health Score:", round(score,2))


In [ ]:
top_category = df.groupby("predicted_category")["amount"].sum().idxmax()
unusual_count = len(df[df["anomaly"]=="Unusual"])

summary = f"""
SMART FINANCIAL REPORT
----------------------
Total Transactions: {len(df)}
Total Spend: {total_spend:.2f}

Highest Spending Category: {top_category}
Unusual Transactions: {unusual_count}

Financial Health Score: {score:.1f}/100

Advice:
{"Your spending pattern is healthy." if score > 60 else "Consider reducing discretionary spending."}
"""

print(summary)


In [ ]:
def predict_transaction(text):
    vec = vectorizer.transform([text])
    return model.predict(vec)[0]

predict_transaction("Amazon online purchase")
